bronze_ingest: Bronze users 2021 con DLT

Tarea 1 del DAG. Este notebook es el respaldo manual si Airflow falla.

ingesta con DLT el Parquet de users del periodo indicado (autores de los posts de ese
periodo) en una ruta fija de Bronze: s3://bronze/users/<periodo>/users_<periodo>.parquet. Sobrescribe
(`replace`), así que repetirlo no crea archivos nuevos.

Por qué lanza un script y no lleva el código dentro: DLT vive en su propio entorno virtual
(`DLT_PYTHON`), creado en el build de la imagen. El kernel de Jupyter no puede importarlo.

Correr con MinIO arriba y Bronze de posts del mismo periodo ya cargado.

In [1]:
import os
import subprocess
import sys

# La carpeta jobs/ esta montada en Jupyter (/workspace/jobs) y en Airflow (/opt/airflow/jobs)
JOBS = os.environ.get("JOBS_DIR") or next(p for p in ("/workspace/jobs", "/opt/airflow/jobs") if os.path.isdir(p))
sys.path.insert(0, JOBS)

import config

# Parametro: periodo de users que se ingesta con DLT (el que corre en Airflow)
PERIOD = "2021_H1"

# DLT vive en su propio entorno virtual (se crea en el build de la imagen, no aqui)
DLT_PYTHON = os.environ["DLT_PYTHON"]

config.period_window(PERIOD)  # valida el periodo
print("Periodo:", PERIOD, "| destino:", f"s3://{config.bronze_path('users', PERIOD)}")
print("Python de DLT:", DLT_PYTHON)

Periodo: 2021_H1 | destino: s3://bronze/users/2021_H1/users_2021_H1.parquet
Python de DLT: /home/airflow/dlt-venv/bin/python


In [2]:
def run_streaming(args):
    """Ejecuta un comando mostrando su salida en vivo; falla si el codigo de salida no es 0."""
    proc = subprocess.Popen(args, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in proc.stdout:
        print(line, end="")
    code = proc.wait()
    if code != 0:
        raise RuntimeError(f"Fallo (codigo {code}): {' '.join(args)}")


# Estado de Bronze ANTES de ingestar (para comparar despues)
run_streaming([DLT_PYTHON, f"{JOBS}/bronze_ls.py"])

    1578.5 MB  bronze/posts/2020_H1/posts_2020_H1.parquet
    1336.9 MB  bronze/posts/2021_H1/posts_2021_H1.parquet
      55.1 MB  bronze/users/2020_H1/users_2020_H1.parquet
3 archivos


In [3]:
# Ingesta con DLT: lee users.parquet de la fuente publica, filtra los autores de los posts del periodo
# y escribe un unico Parquet en una ruta fija de Bronze (write_disposition="replace").
run_streaming([DLT_PYTHON, f"{JOBS}/bronze_users_dlt.py", PERIOD])

/home/airflow/dlt-venv/lib/python3.10/site-packages/dlt/helpers/dbt/__init__.py:3: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources
users 2021_H1: 599150 autores distintos; datasets-documentation/stackoverflow/parquet/users.parquet
  22484235 filas en origen, 27 row groups; lotes de 300000
  300000/22484235 leidas (1%), 10473 conservadas, 40s, RAM pico 597 MB
  600000/22484235 leidas (3%), 19358 conservadas, 41s, RAM pico 711 MB
  900000/22484235 leidas (4%), 31704 conservadas, 41s, RAM pico 751 MB
  1200000/22484235 leidas (5%), 39010 conservadas, 199s, RAM pico 833 MB
  1500000/22484235 leidas (7%), 48694 conservadas, 199s, RAM pico 844 MB
  1800000/22484235 leidas (8%), 55687 conservadas, 199s, RAM pico 860 MB
  2100000/22484235 leidas (9%), 67793 conservadas, 243s, RAM 

In [4]:
# Estado de Bronze DESPUES: debe existir users/<periodo>/ y, si repites la ejecucion, el MISMO numero de archivos
run_streaming([DLT_PYTHON, f"{JOBS}/bronze_ls.py"])
print()
# Contenido: filas, fechas y cobertura de users frente a los autores de posts
run_streaming([DLT_PYTHON, f"{JOBS}/bronze_check.py"])

    1578.5 MB  bronze/posts/2020_H1/posts_2020_H1.parquet
    1336.9 MB  bronze/posts/2021_H1/posts_2021_H1.parquet
      55.1 MB  bronze/users/2020_H1/users_2020_H1.parquet
      48.7 MB  bronze/users/2021_H1/users_2021_H1.parquet
       0.0 MB  bronze/users/_dlt_loads/bronze_users__1791724113.1287146.jsonl
       0.0 MB  bronze/users/_dlt_loads/init
       0.0 MB  bronze/users/_dlt_pipeline_state/bronze_users__1791724113.1287146__02e31c5e2b2337ada3bbb8f17461e3b8fcd7f48a67f9f4853f8c877df195fcb5.jsonl
       0.0 MB  bronze/users/_dlt_pipeline_state/init
       0.0 MB  bronze/users/_dlt_version/bronze_users__1791724528.914058__02e31c5e2b2337ada3bbb8f17461e3b8fcd7f48a67f9f4853f8c877df195fcb5.jsonl
       0.0 MB  bronze/users/_dlt_version/init
       0.0 MB  bronze/users/init
11 archivos

[posts 2020_H1] 2283275 filas, 22 columnas, 15 row groups
    fechas 2020-01-01 .. 2020-06-30 -> dentro del periodo: SI
    autores distintos (OwnerUserId > 0): 665402 | PostTypeId: {1: 993429, 2: 128766